# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
# DEBIAN_FRONTEND=noninteractive so any apt post-install prompt
# doesn't stall the whole boot silently (fontconfig sometimes prompts
# on a fresh Kaggle image).
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.check_call(['apt-get', '-qq', 'update'])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai). DejaVu Sans is the primary font (bundled default)
# with libass falling back to Noto CJK for CJK. -y and DEBIAN_FRONTEND
# ensure no interactive prompt.
subprocess.check_call(['apt-get', '-qq', 'install', '-y',
                       '--no-install-recommends',
                       'ffmpeg', 'fonts-dejavu-core',
                       'fonts-noto', 'fonts-noto-cjk',
                       'fonts-noto-color-emoji'])
subprocess.run(['fc-cache', '-f'], check=False)

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — split into small pip calls with diagnostic prints
#    so any hang/failure shows up in the Kaggle log with a specific
#    step tag. Previous 2026-07-11 boot failure gave no diagnostic
#    because the whole install was one opaque call.
import subprocess, sys, os, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

def _step(tag):
    print(f'>>> [{tag}] start ts={time.strftime("%H:%M:%S")}', flush=True)
    return time.time()

def _done(tag, t0):
    print(f'<<< [{tag}] done in {time.time()-t0:.1f}s', flush=True)

def _pip(tag, *args, allow_fail=False):
    t0 = _step(f'pip {tag}')
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--no-input'] + list(args)
    try:
        subprocess.check_call(cmd)
        _done(f'pip {tag}', t0)
        return True
    except subprocess.CalledProcessError as e:
        print(f'!!! [pip {tag}] failed rc={e.returncode}', flush=True)
        if not allow_fail:
            raise
        return False

# apt: espeak-ng for phonemizer (Kokoro dep)
t0 = _step('apt espeak-ng')
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])
_done('apt espeak-ng', t0)

# Remove stock phonemizer so phonemizer-fork can install cleanly later.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)

# Base project requirements first (transformers>=4.44, diffusers>=0.30 floors).
_pip('base-requirements', '-r', 'requirements.txt')
_pip('browser-requirements', '-r', 'requirements-browser.txt')

# GPU extras: bumps to transformers>=4.55 + diffusers>=0.36 for klein-4B,
# torch pin (cu121), decord, av, kokoro, misaki.
_pip('gpu-requirements', '-r', 'requirements-gpu.txt')

# Explicit klein-4B stack — force the versions in case a transitive
# dep tried to hold something down.
_pip('klein4b-stack', '--upgrade',
     'diffusers>=0.36,<0.40',
     'transformers>=4.55,<5.0',  # bumped from 4.51 — see requirements-gpu.txt for reason
     'sentencepiece>=0.2',
     'accelerate>=0.30',
     'safetensors>=0.4',
     'huggingface_hub>=0.23',
     'hf_transfer',
     'peft>=0.10')

# Kokoro — try 1.x first (Qwen3-era, transformers>=4.55 compatible).
# If pip resolver rejects (e.g. no 1.x release yet, or missing wheel),
# fall back to 0.9.x with --no-deps so it doesn't drag transformers
# back down. In the worst case Kokoro fails at runtime import and
# modules/voiceover.py falls back to edge-tts automatically.
if not _pip('kokoro-1x', '--upgrade', 'kokoro>=1.0', 'misaki>=0.9.4', allow_fail=True):
    print('kokoro>=1.0 unavailable — falling back to 0.9.x --no-deps', flush=True)
    _pip('kokoro-0.9-nodeps', '--no-deps', '--upgrade', 'kokoro>=0.9.4', 'misaki>=0.9.4')

# TLS / crypto — pinning avoids the GEN_EMAIL crash on some jobs.
_pip('crypto', 'pyOpenSSL>=25.0.0', 'cryptography>=45.0.0')

# phonemizer-fork replaces the removed phonemizer.
_pip('phonemizer-fork', '--force-reinstall', 'phonemizer-fork>=3.3.2')
_pip('espeakng-loader', 'espeakng-loader>=0.2', 'soundfile>=0.12')

# edge-tts fallback for languages Kokoro can't do (and if Kokoro breaks).
_pip('edge-tts', '--upgrade', 'edge-tts>=7.0.0')

# Chromium install runs in the background — this cell exits without waiting.
_playwright_log = open('/tmp/playwright-install.log', 'w')
_playwright_bg = subprocess.Popen(
    [sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
    stdout=_playwright_log, stderr=subprocess.STDOUT,
)
print(f'playwright chromium install running in background (pid={_playwright_bg.pid})', flush=True)

# ================= import diagnostics =================
# Every check is printed so ANY failure has a stable stderr signature
# we can grep for in the Kaggle log.
print('\n=== IMPORT DIAGNOSTICS ===', flush=True)
import torch
print(f'torch: {torch.__version__} cuda_available: {torch.cuda.is_available()}', flush=True)

for _mod in ['torchvision', 'transformers', 'diffusers', 'accelerate', 'sentencepiece']:
    try:
        _m = __import__(_mod)
        print(f'{_mod}: {getattr(_m, "__version__", "installed")}', flush=True)
    except Exception as e:
        print(f'{_mod}: IMPORT-FAIL -> {e!r}', flush=True)

# Kokoro — non-fatal. If it breaks, edge-tts takes over.
try:
    from kokoro import KPipeline  # noqa: F401
    print('kokoro: ready', flush=True)
except Exception as e:
    print(f'kokoro: IMPORT-FAIL -> {e!r} (falling back to edge-tts at runtime)', flush=True)

# SDXL pipeline — must load for the SDXL fallback provider.
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers AutoPipeline: ready (local_sdxl available)', flush=True)
except Exception as e:
    print(f'diffusers AutoPipeline: IMPORT-FAIL -> {e!r}', flush=True)

# Qwen3ForCausalLM probe — klein-4B's text encoder needs this
# class registered on transformers's model registry. Was missing
# on 4.51.x (see 2026-07-11 boot log). This tells us straight up
# whether the bump did what we hoped.
try:
    from transformers import Qwen3ForCausalLM  # noqa: F401
    print('transformers Qwen3ForCausalLM: registered (klein-4B text encoder can load)', flush=True)
except Exception as e:
    print(f'transformers Qwen3ForCausalLM: NOT REGISTERED -> {e!r}', flush=True)
    print('  --> klein-4B will still fail. Try bumping transformers even higher.', flush=True)

# Klein-4B pipeline — THE reason for the upgrade. Must succeed for
# local_flux2_klein to be available.
try:
    from diffusers import Flux2KleinPipeline  # noqa: F401
    print('diffusers Flux2KleinPipeline: ready (local_flux2_klein AVAILABLE)', flush=True)
except Exception as e:
    print(f'diffusers Flux2KleinPipeline: IMPORT-FAIL -> {e!r}', flush=True)
    print('  --> klein-4B provider will auto-disable; chain falls to Pollinations/SDXL', flush=True)

# edge-tts always required as a fallback.
try:
    import edge_tts
    print(f'edge-tts: {getattr(edge_tts, "__version__", "installed")}', flush=True)
except Exception as e:
    print(f'edge-tts: IMPORT-FAIL -> {e!r}', flush=True)

try:
    import OpenSSL, cryptography
    print(f'crypto: pyOpenSSL={OpenSSL.__version__} cryptography={cryptography.__version__}', flush=True)
except Exception as e:
    print(f'crypto: IMPORT-FAIL -> {e!r}', flush=True)

print('=== IMPORT DIAGNOSTICS DONE ===\n', flush=True)


In [ ]:
# 3) BOOTSTRAP secrets.
#
# Two sources, in priority order:
#
#   (1) A private Kaggle Dataset attached via kernel-metadata.json. This
#       is the production path — survives `kaggle kernels push` cycles
#       (each push creates a new kernel version, but the dataset stays
#       attached). Create one private Dataset named e.g.
#       "yt-agent-secrets" with ONE file:
#
#           secrets.env
#               COOLIFY_BASE_URL=https://yt-agent.thyker.online
#               PB_URL=https://yt-agent.thyker.online/pb
#               POCKETBASE_ADMIN_EMAIL=admin@yt-agent.thyker.online
#               POCKETBASE_ADMIN_PASSWORD=your-password
#               RENDER_TRIGGER_KEY=...
#               STORAGE_PROVIDERS_ENC_KEY=...
#
#       Then reference it in kernel-metadata.json:
#           "dataset_sources": ["<your-username>/yt-agent-secrets"]
#
#   (2) Kaggle's Secrets panel (Add-ons → Secrets). Useful for
#       interactive testing. Same key names as the .env file.
#
# Either source works — the notebook reads whichever it finds first.
import os, glob

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    # Legacy Vercel + Firestore path — keep working for users who
    # haven\'t migrated to Coolify yet.
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]
ALL_KEYS = REQUIRED + OPTIONAL


def _load_from_dataset_envfile():
    """Look for an attached Dataset secrets file. Kaggle mounts datasets
    under /kaggle/input/<dataset-name>/. Search for the most common
    file names."""
    candidates = []
    for root in glob.glob('/kaggle/input/*/'):
        for name in ('secrets.env', '.env', 'env.txt', 'secrets.txt'):
            p = os.path.join(root, name)
            if os.path.exists(p):
                candidates.append(p)
    if not candidates:
        return False
    found = False
    for path in candidates:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ALL_KEYS and v:
                    os.environ.setdefault(k, v)
                    found = True
        print(f'  loaded secrets from {path}')
    return found


def _load_from_kaggle_secrets():
    """Read from the Kaggle Secrets panel. Fails silently when not
    available (e.g. running in a Colab clone of the notebook)."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        loaded = 0
        for k in ALL_KEYS:
            try:
                v = secrets.get_secret(k)
            except Exception:
                continue
            if v and not os.environ.get(k):
                os.environ[k] = v
                loaded += 1
        if loaded:
            print(f'  loaded {loaded} secrets from Kaggle Secrets panel')
        return loaded > 0
    except Exception as e:
        print(f'  Kaggle Secrets unavailable ({e!r}); using Dataset only')
        return False


# Try both sources — Dataset first, then Kaggle Secrets fills any gaps.
got_dataset = _load_from_dataset_envfile()
got_secrets = _load_from_kaggle_secrets()
if not (got_dataset or got_secrets):
    raise SystemExit(
        'No secrets source found. Either attach the yt-agent-secrets '
        'Dataset (via kernel-metadata.json dataset_sources) OR add the '
        'required keys to the Kaggle Secrets panel.'
    )

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))

# Defaults for the new outbound-poll mode.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Idle auto-shutdown ~25 min — preserves the 30 GPU-hr/week budget.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

print('Kaggle bootstrap OK')
print('  Dashboard:', os.environ.get('COOLIFY_BASE_URL'))
print('  Pocketbase:', os.environ.get('PB_URL'))
print('  Mode:', os.environ.get('WORKER_MODE'), '| tier:', os.environ.get('INSTANCE_TIER'))


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 4.5) Pre-warm image-gen weights in the BACKGROUND — doesn't block boot.
#
# Two parallel snapshot threads:
#   - SDXL fp16 (~7 GB) — final fallback provider.
#   - klein-4B (~8 GB) — primary local provider when CF pool exhausts.
#
# Both use HF_HUB_ENABLE_HF_TRANSFER=1 for fast parallel downloads.
# If klein-4B's diffusers/transformers upgrade failed in cell 2, its
# provider is auto-disabled — we still kick the snapshot thread so the
# weights are cached for a future retry with a fixed cell 2. The
# thread crash from an unavailable model repo is caught below.
import os, time, subprocess, threading
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '180')

def _bg_snapshot_sdxl():
    try:
        t0 = time.time()
        print('preboot(bg): snapshotting SDXL fp16 weights into HF cache...', flush=True)
        from huggingface_hub import snapshot_download
        _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
        snapshot_download(
            _sdxl_model,
            allow_patterns=[
                '*.json', '*.txt',
                '**/*.fp16.safetensors',
                'sd_xl_turbo_1.0_fp16.safetensors',
            ],
        )
        print(f'preboot(bg): SDXL fp16 weights cached in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'preboot(bg): SDXL cache warm skipped ({e!r})', flush=True)

def _bg_snapshot_flux2_klein():
    try:
        t0 = time.time()
        print('preboot(bg): snapshotting klein-4B weights into HF cache...', flush=True)
        from huggingface_hub import snapshot_download
        _model = os.environ.get('LOCAL_FLUX2_KLEIN_MODEL', 'black-forest-labs/FLUX.2-klein-4B')
        snapshot_download(
            _model,
            allow_patterns=[
                '*.json', '*.txt',
                'scheduler/*.json',
                'text_encoder/*.safetensors',
                'transformer/*.safetensors',
                'vae/*.safetensors',
                'tokenizer/*',
            ],
        )
        print(f'preboot(bg): klein-4B weights cached in {time.time()-t0:.1f}s', flush=True)
    except Exception as e:
        print(f'preboot(bg): klein-4B cache warm skipped ({e!r})', flush=True)

threading.Thread(target=_bg_snapshot_sdxl, name='sdxl-snapshot', daemon=True).start()
threading.Thread(target=_bg_snapshot_flux2_klein, name='klein4b-snapshot', daemon=True).start()
print('preboot: SDXL + klein-4B snapshots kicked in background threads', flush=True)

try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out, flush=True)
    if out.count('GPU ') < 2:
        print('preboot: WARNING - only 1 GPU visible. To use T4x2, set '
              'Notebook Settings -> Accelerator -> GPU T4 x2 -> Save Version.',
              flush=True)
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})', flush=True)

print('preboot: done (main thread not blocked on downloads)', flush=True)


In [ ]:
# 4.6) Self-check — verify multi-GPU + multilingual wiring
#
# Wrapped in try/except so any check crash still lets uvicorn boot.
# The self-check is diagnostic, not a gate.
try:
    from backend import self_check
    print(self_check.run(text=True), flush=True)
except Exception as e:
    print(f'self_check skipped ({e!r})', flush=True)

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])